# Validación de la experimentación offline

Este cuaderno verifica configuración, versiones, disponibilidad, integridad y consultas de las tres fuentes utilizadas. El alcance offline queda limitado a recuperación de una vista, detección de objetos, recuperación informada por objetos y contraste aislado de relaciones 2D. La multivista, las habitaciones y la navegación se evaluarán en simulación.

In [1]:
from pathlib import Path
import sys
repo = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'semantic_navigation_ws' / 'src').is_dir())
sys.path.insert(0, str(repo / 'experiments' / 'shared'))
import importlib.metadata as metadata
import pandas as pd
from notebook_bootstrap import bootstrap_offline, resolve_repo_path
from offline_benchmarks import build_query_catalog, create_data_figures, summarize_query_catalog
from semantic_evaluation.core.config_validation import validate_offline_isolation
from semantic_evaluation.core.dataset_adapters import load_dataset, validate_dataset
from semantic_evaluation.core.offline_dataset import load_queries, validate_queries
ctx = bootstrap_offline()
config = ctx['config']
validate_offline_isolation(config, str(ctx['config_path']))
print(f"Configuración: {ctx['config_path']}")
print(f"Dispositivo: {ctx['device']} · semilla: {config['experiment']['seed']}")

/home/junior/visual_semantic_navigation/.venv-1/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Configuración: /home/junior/visual_semantic_navigation/experiments/offline/config/offline_experiment_config.yaml
Dispositivo: cuda · semilla: 42


## Entorno, modelos y datasets

In [2]:
packages = ['numpy', 'pandas', 'torch', 'transformers', 'ultralytics', 'Pillow', 'matplotlib']
versions = []
for package in packages:
    try:
        version = metadata.version(package)
    except metadata.PackageNotFoundError:
        version = 'no instalado'
    versions.append({'package': package, 'version': version})
display(pd.DataFrame(versions))
display(pd.DataFrame([
    {'family': 'VLM', 'variant': key, 'checkpoint': value}
    for key, value in config['models']['siglip']['variants'].items()
] + [
    {'family': 'detector', 'variant': key, 'checkpoint': value}
    for key, value in config['models']['yolo']['variants'].items()
]))
bundles = {}
issue_rows = []
for spec in ctx['dataset_specs']:
    bundle = load_dataset(spec, ctx['repo_root'])
    bundles[spec.dataset_id] = bundle
    issue_rows.extend({'dataset_id': spec.dataset_id, **issue} for issue in validate_dataset(bundle))
availability = pd.DataFrame([
    {'dataset_id': key, 'split': value.split, 'available': not value.skipped,
     'nodes': len(value.nodes), 'object_annotations': len(value.object_ground_truth),
     'relation_annotations': len(value.relation_ground_truth),
     'status': value.skip_reason or 'available'}
    for key, value in bundles.items()
])
display(availability)
display(pd.DataFrame(issue_rows))

,package,version
0,numpy,1.26.4
1,pandas,3.0.2
2,torch,2.11.0
3,transformers,5.3.0
4,ultralytics,8.4.39
5,Pillow,12.1.1
6,matplotlib,3.10.8


,family,variant,checkpoint
0,VLM,siglip_v1,google/siglip-base-patch16-224
1,VLM,siglip_v2,google/siglip2-base-patch16-224
2,detector,yolov8n,experiments/yolov8n.pt
3,detector,yolo26n,experiments/yolo26n.pt


,dataset_id,split,available,nodes,object_annotations,relation_annotations,status
0,siglip_rooms,validation,True,300,0,0,available
1,sunrgbd,validation,True,2619,2619,0,available
2,visual_genome,validation,True,0,96444,96444,available


,dataset_id,severity,item_id,message
0,siglip_rooms,warning,bedroom_0020:view_000,duplicate image content of bedroom_0011:view_000
1,siglip_rooms,warning,livingroom_0026:view_000,duplicate image content of livingroom_0008:vie...


## Consultas, muestras visuales y exportación

Las consultas también son entradas del experimento: se documentan su texto, idioma, longitud, contenido semántico y alcance. Se distingue entre consultas generales que nombran una estancia, descripciones implícitas o funcionales, consultas específicas basadas en objetos y negativas. Las imágenes mostradas son ejemplos reproducibles del conjunto propio y de SUN RGB-D; Visual Genome se utiliza localmente solo mediante anotaciones.

In [3]:
from reproducibility import collect_manifest, save_manifest
query_rows = []
query_issues = []
manifest_root = resolve_repo_path(ctx['repo_root'], config['paths']['manifests_root'])
results_root = resolve_repo_path(ctx['repo_root'], config['paths']['results_root']) / 'data_validation'
manifest_root.mkdir(parents=True, exist_ok=True)
results_root.mkdir(parents=True, exist_ok=True)
query_catalog = build_query_catalog(ctx)
query_profile_summary = summarize_query_catalog(query_catalog)
for spec in ctx['dataset_specs']:
    bundle = bundles[spec.dataset_id]
    queries = load_queries(str(resolve_repo_path(ctx['repo_root'], spec.queries_file)))
    if not bundle.skipped and bundle.nodes:
        query_issues.extend({'dataset_id': spec.dataset_id, 'issue': issue}
                            for issue in validate_queries(queries, bundle))
    query_rows.append({'dataset_id': spec.dataset_id, 'queries': len(queries)})
    manifest = collect_manifest(config, repo_dir=str(ctx['repo_root']), device=ctx['device'],
        extra={'notebook': '00_data_validation', 'dataset_id': spec.dataset_id,
               'item_ids': bundle.node_ids(),
               'status': 'skipped' if bundle.skipped else 'validated',
               'skip_reason': bundle.skip_reason})
    save_manifest(str(manifest_root / f'{spec.dataset_id}.json'), manifest)
queries_summary = pd.DataFrame(query_rows)
availability.to_csv(results_root / 'dataset_availability.csv', index=False)
queries_summary.to_csv(results_root / 'query_counts.csv', index=False)
query_catalog.to_csv(results_root / 'query_catalog.csv', index=False)
query_profile_summary.to_csv(results_root / 'query_profile_summary.csv', index=False)
pd.DataFrame(issue_rows + query_issues).to_csv(results_root / 'validation_issues.csv', index=False)
figure_paths = create_data_figures(bundles, query_catalog, results_root / 'figures')
display(queries_summary)
display(query_profile_summary)
display(pd.DataFrame(query_issues))
if not availability['available'].all():
    raise RuntimeError('Hay datasets offline obligatorios no disponibles.')
print(f'Validación guardada en {results_root}')
print('Figuras generadas:')
for path in figure_paths:
    print(' -', path)

,dataset_id,queries
0,siglip_rooms,13
1,sunrgbd,58
2,visual_genome,0


,dataset_id,language,scope,formulation,n_queries,mean_words,min_words,max_words,bilingual_pairs
0,siglip_rooms,en,descriptive,functional_description,1,12.000000,12,12,1
1,siglip_rooms,en,general,direct_room,1,4.000000,4,4,1
2,siglip_rooms,en,not_applicable,negative,1,4.000000,4,4,0
3,siglip_rooms,en,specific,multi_object_description,1,8.000000,8,8,0
4,siglip_rooms,en,specific,object_description,1,5.000000,5,5,1
5,siglip_rooms,es,descriptive,attribute_description,1,7.000000,7,7,0
6,siglip_rooms,es,descriptive,functional_description,1,8.000000,8,8,0
7,siglip_rooms,es,descriptive,paraphrase,1,10.000000,10,10,0
8,siglip_rooms,es,general,direct_room,2,4.500000,4,5,1
9,siglip_rooms,es,not_applicable,negative,1,4.000000,4,4,0


""


Validación guardada en /home/junior/visual_semantic_navigation/experiments/offline/results/data_validation
Figuras generadas:
 - /home/junior/visual_semantic_navigation/experiments/offline/results/data_validation/figures/datasets_ejemplos.png
 - /home/junior/visual_semantic_navigation/experiments/offline/results/data_validation/figures/datasets_ejemplos.pdf
 - /home/junior/visual_semantic_navigation/experiments/offline/results/data_validation/figures/consultas_composicion.png
 - /home/junior/visual_semantic_navigation/experiments/offline/results/data_validation/figures/consultas_composicion.pdf
